<a href="https://colab.research.google.com/github/Likith-Reddy25/Summer-Intern/blob/main/codes/MNIST_Dedicated_24.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q \
    qiskit==1.1.0 \
    qiskit-machine-learning==0.7.2 \
    qiskit-algorithms==0.3.0 \
    scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.8/97.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.6/308.6 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.3/50.3 MB 18.6 MB/s eta 0:00:00


In [ ]:


import numpy as np
import pandas as pd
import time
import warnings

warnings.filterwarnings("ignore")

from sklearn.datasets import fetch_openml
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    f1_score
)

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import ZZFeatureMap

from qiskit_machine_learning.kernels import (
    FidelityStatevectorKernel,
    TrainableFidelityStatevectorKernel
)

from qiskit_machine_learning.kernels.algorithms import (
    QuantumKernelTrainer
)

from qiskit_machine_learning.utils.loss_functions import (
    SVCLoss
)

from qiskit_algorithms.optimizers import SPSA


# ============================================================
# EXPERIMENT SETTINGS
# ============================================================

BASE_SEED = 12345

N_FEATURES = 8

TRAIN_PER_CLASS = 125
TEST_PER_CLASS = 125

# Final paper experiment
N_REPETITIONS = 5
MAX_ITER = 50

# Uncomment these for quick testing
# N_REPETITIONS = 3
# MAX_ITER = 30

C_VALUES = [
    0.01,
    0.1,
    1,
    10,
    100
]

LAMBDA_VALUES = [
    0.001,
    0.01,
    0.1,
    0.5,
    1.0
]

print("="*60)
print("QKT Dedicated (24)")
print("="*60)
print("Features          :", N_FEATURES)
print("Train size        :", 250)
print("Test size         :", 250)
print("Repetitions       :", N_REPETITIONS)
print("SPSA Iterations   :", MAX_ITER)
print("="*60)

QKT Dedicated (24)
Features          : 8
Train size        : 250
Test size         : 250
Repetitions       : 5
SPSA Iterations   : 50


In [ ]:
# ============================================================
# BLOCK 2 : DATA PREPARATION
# MNIST -> PCA(8)
# ============================================================

print("Loading MNIST...")

mnist = fetch_openml(
    "mnist_784",
    version=1,
    as_frame=False
)

X = mnist.data.astype(np.float64)
y = mnist.target.astype(int)

# ------------------------------------------------------------
# Keep only digits 3 and 5
# ------------------------------------------------------------

mask = np.isin(y, [3, 5])

X = X[mask]
y = y[mask]

# Normalize pixels
X = X / 255.0

# Labels
# 3 -> -1
# 5 -> +1

y = np.where(y == 3, -1, 1)

print("Filtered Dataset Shape :", X.shape)
print("Class Distribution")

print("Digit 3 (-1):", np.sum(y == -1))
print("Digit 5 (+1):", np.sum(y == 1))


# ------------------------------------------------------------
# PCA -> 8 Components
# ------------------------------------------------------------

print("\nApplying PCA...")

pca = PCA(
    n_components=N_FEATURES,
    random_state=BASE_SEED
)

X_pca = pca.fit_transform(X)

print("PCA Shape :", X_pca.shape)

print(
    "Explained Variance :",
    round(
        pca.explained_variance_ratio_.sum(),
        4
    )
)


# ============================================================
# BALANCED TRAIN / TEST SPLIT
# 125 samples/class for training
# 125 samples/class for testing
# ============================================================

def create_balanced_split(
    X,
    y,
    seed
):

    rng = np.random.default_rng(seed)

    negative = np.where(y == -1)[0]
    positive = np.where(y == 1)[0]

    rng.shuffle(negative)
    rng.shuffle(positive)

    train_idx = np.concatenate([
        negative[:TRAIN_PER_CLASS],
        positive[:TRAIN_PER_CLASS]
    ])

    test_idx = np.concatenate([
        negative[
            TRAIN_PER_CLASS:
            TRAIN_PER_CLASS + TEST_PER_CLASS
        ],
        positive[
            TRAIN_PER_CLASS:
            TRAIN_PER_CLASS + TEST_PER_CLASS
        ]
    ])

    rng.shuffle(train_idx)
    rng.shuffle(test_idx)

    X_train = X[train_idx]
    X_test = X[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    return (
        X_train,
        X_test,
        y_train,
        y_test
    )


# ============================================================
# METRIC FUNCTION
# ============================================================

def calculate_metrics(
    y_true,
    y_pred
):

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    kappa = cohen_kappa_score(
        y_true,
        y_pred
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro"
    )

    return (
        accuracy,
        kappa,
        macro_f1
    )


print("\nData preparation complete.")

Loading MNIST...
Filtered Dataset Shape : (13454, 784)
Class Distribution
Digit 3 (-1): 7141
Digit 5 (+1): 6313

Applying PCA...
PCA Shape : (13454, 8)
Explained Variance : 0.4747

Data preparation complete.


In [ ]:
# ============================================================
# BLOCK 3 : QKT Dedicated (24) Circuit
# ============================================================

def create_dedicated24_circuit(n_features=8):

    # --------------------------------------------------------
    # 24 trainable parameters
    # (3 parameters per qubit)
    # --------------------------------------------------------

    theta = ParameterVector(
        "theta",
        length=3 * n_features
    )


    # --------------------------------------------------------
    # Dedicated trainable layer
    # --------------------------------------------------------

    dedicated_layer = QuantumCircuit(
        n_features
    )


    for qubit in range(n_features):

        dedicated_layer.u(

            theta[3 * qubit],

            theta[3 * qubit + 1],

            theta[3 * qubit + 2],

            qubit

        )


    # --------------------------------------------------------
    # ZZ Feature Map
    # --------------------------------------------------------

    feature_map = ZZFeatureMap(

        feature_dimension=n_features,

        reps=2

    )


    # --------------------------------------------------------
    # Complete QKT Circuit
    #
    # |ψ(x,θ)> = Uθ · Uφ(x) |0>
    # --------------------------------------------------------

    circuit = QuantumCircuit(
        n_features
    )

    circuit.compose(
        dedicated_layer,
        inplace=True
    )

    circuit.compose(
        feature_map,
        inplace=True
    )

    return circuit, list(theta)

In [ ]:
# ============================================================
# BLOCK 4 : Hyperparameter Search (One-Time)
# Finds BEST_LAMBDA and BEST_C
# ============================================================

print("=" * 60)
print("Searching for Best λ and C")
print("=" * 60)

# ------------------------------------------------------------
# Create one reference train/test split
# ------------------------------------------------------------

X_train_ref, X_test_ref, y_train_ref, y_test_ref = create_balanced_split(
    X_pca,
    y,
    BASE_SEED
)

# ------------------------------------------------------------
# Scale to [0, 2π]
# ------------------------------------------------------------

scaler = MinMaxScaler(
    feature_range=(0, 2 * np.pi)
)

X_train_ref = scaler.fit_transform(X_train_ref)
X_test_ref = scaler.transform(X_test_ref)

# ------------------------------------------------------------
# Cross Validation
# ------------------------------------------------------------

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=BASE_SEED
)

splits = list(
    cv.split(
        X_train_ref,
        y_train_ref
    )
)

# ------------------------------------------------------------
# QKE Feature Map
# ------------------------------------------------------------

feature_map = ZZFeatureMap(
    feature_dimension=N_FEATURES,
    reps=2
)

kernel = FidelityStatevectorKernel(
    feature_map=feature_map
)

BEST_SCORE = -1
BEST_LAMBDA = None
BEST_C = None

# ------------------------------------------------------------
# Grid Search
# ------------------------------------------------------------

for lam in LAMBDA_VALUES:

    print(f"\nλ = {lam}")

    X_train_lambda = X_train_ref * lam

    # Compute kernel only once for current λ
    K = kernel.evaluate(
        x_vec=X_train_lambda
    )

    for C in C_VALUES:

        fold_scores = []

        for train_idx, val_idx in splits:

            K_train = K[
                np.ix_(
                    train_idx,
                    train_idx
                )
            ]

            K_val = K[
                np.ix_(
                    val_idx,
                    train_idx
                )
            ]

            model = SVC(
                kernel="precomputed",
                C=C
            )

            model.fit(
                K_train,
                y_train_ref[train_idx]
            )

            prediction = model.predict(
                K_val
            )

            score = accuracy_score(
                y_train_ref[val_idx],
                prediction
            )

            fold_scores.append(score)

        mean_score = np.mean(fold_scores)

        print(
            f"   C = {C:<6}  CV Accuracy = {mean_score:.4f}"
        )

        if mean_score > BEST_SCORE:

            BEST_SCORE = mean_score
            BEST_LAMBDA = lam
            BEST_C = C


print("\n" + "=" * 60)
print("Best Hyperparameters")
print("=" * 60)

print("Best λ :", BEST_LAMBDA)
print("Best C :", BEST_C)
print("Best CV Accuracy :", round(BEST_SCORE, 4))

Searching for Best λ and C

λ = 0.001
   C = 0.01    CV Accuracy = 0.6320
   C = 0.1     CV Accuracy = 0.6320
   C = 1       CV Accuracy = 0.6320
   C = 10      CV Accuracy = 0.6840
   C = 100     CV Accuracy = 0.8800

λ = 0.01
   C = 0.01    CV Accuracy = 0.7240
   C = 0.1     CV Accuracy = 0.7400
   C = 1       CV Accuracy = 0.9200
   C = 10      CV Accuracy = 0.9360
   C = 100     CV Accuracy = 0.9520

λ = 0.1
   C = 0.01    CV Accuracy = 0.7200
   C = 0.1     CV Accuracy = 0.7200
   C = 1       CV Accuracy = 0.7520
   C = 10      CV Accuracy = 0.7640
   C = 100     CV Accuracy = 0.7640

λ = 0.5
   C = 0.01    CV Accuracy = 0.6160
   C = 0.1     CV Accuracy = 0.6160
   C = 1       CV Accuracy = 0.6240
   C = 10      CV Accuracy = 0.6080
   C = 100     CV Accuracy = 0.6080

λ = 1.0
   C = 0.01    CV Accuracy = 0.5200
   C = 0.1     CV Accuracy = 0.5200
   C = 1       CV Accuracy = 0.5000
   C = 10      CV Accuracy = 0.5000
   C = 100     CV Accuracy = 0.5000

Best Hyperparameters
Bes

In [ ]:
# ============================================================
# BLOCK 5A : Train Dedicated (24) Kernel
# ============================================================

def train_dedicated24_kernel(
    X_train,
    y_train,
    C_value,
    max_iter
):

    # --------------------------------------------------------
    # Create Dedicated (24) circuit
    # --------------------------------------------------------

    circuit, training_parameters = create_dedicated24_circuit(
        N_FEATURES
    )

    # --------------------------------------------------------
    # Trainable Quantum Kernel
    # --------------------------------------------------------

    trainable_kernel = TrainableFidelityStatevectorKernel(

        feature_map=circuit,

        training_parameters=training_parameters

    )

    # --------------------------------------------------------
    # SPSA Optimizer
    # --------------------------------------------------------

    optimizer = SPSA(

        maxiter=max_iter,

        second_order=False,

        blocking=True
    )

    # For paper reproduction later, change to:
    #
    # optimizer = SPSA(
    #     maxiter=max_iter,
    #     second_order=True,
    #     blocking=True
    # )

    # --------------------------------------------------------
    # Loss Function
    # --------------------------------------------------------

    loss = SVCLoss(
        C=C_value
    )

    # --------------------------------------------------------
    # Quantum Kernel Trainer
    # --------------------------------------------------------

    trainer = QuantumKernelTrainer(

        quantum_kernel=trainable_kernel,

        loss=loss,

        optimizer=optimizer,

        initial_point=np.zeros(24)

    )

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    result = trainer.fit(

        X_train,

        y_train

    )

    return (

        result.quantum_kernel,

        result.optimal_point,

        result.optimal_value

    )

In [ ]:
# ============================================================
# BLOCK 5B-1 : Main Repetition Loop
# Dedicated (24)
# ============================================================

dedicated_results = []

dedicated_parameters = []

for rep in range(N_REPETITIONS):

    print("\n" + "=" * 70)
    print(f"DEDICATED (24) : REPETITION {rep+1}/{N_REPETITIONS}")
    print("=" * 70)

    start_time = time.time()

    seed = BASE_SEED + rep

    # --------------------------------------------------------
    # Balanced Train/Test Split
    # --------------------------------------------------------

    (
        X_train,
        X_test,
        y_train,
        y_test
    ) = create_balanced_split(
        X_pca,
        y,
        seed
    )

    # --------------------------------------------------------
    # Scale to [0,2π]
    # --------------------------------------------------------

    scaler = MinMaxScaler(
        feature_range=(0, 2*np.pi)
    )

    X_train = scaler.fit_transform(
        X_train
    )

    X_test = scaler.transform(
        X_test
    )

    # --------------------------------------------------------
    # Apply Selected Lambda
    # --------------------------------------------------------

    X_train = X_train * BEST_LAMBDA

    X_test = X_test * BEST_LAMBDA

    print(f"Lambda : {BEST_LAMBDA}")
    print(f"C      : {BEST_C}")

    # --------------------------------------------------------
    # Train Dedicated (24) Kernel
    # --------------------------------------------------------

    print("\nTraining Dedicated (24)...")

    (
        optimized_kernel,
        optimal_theta,
        optimal_loss
    ) = train_dedicated24_kernel(

        X_train,

        y_train,

        BEST_C,

        MAX_ITER

    )

    print("\nOptimization Complete")

    print("Loss :", optimal_loss)

    print("Theta Shape :", len(optimal_theta))
        # --------------------------------------------------------
    # Compute Optimized Training Kernel
    # --------------------------------------------------------

    print("\nComputing Training Kernel...")

    K_train = optimized_kernel.evaluate(
        x_vec=X_train
    )

    # --------------------------------------------------------
    # Compute Optimized Test Kernel
    # --------------------------------------------------------

    print("Computing Test Kernel...")

    K_test = optimized_kernel.evaluate(
        x_vec=X_test,
        y_vec=X_train
    )

    # --------------------------------------------------------
    # Train SVM
    # --------------------------------------------------------

    svm = SVC(
        kernel="precomputed",
        C=BEST_C
    )

    svm.fit(
        K_train,
        y_train
    )

    # --------------------------------------------------------
    # Predictions
    # --------------------------------------------------------

    train_prediction = svm.predict(
        K_train
    )

    test_prediction = svm.predict(
        K_test
    )

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    acc_tr, kappa_tr, mf_tr = calculate_metrics(
        y_train,
        train_prediction
    )

    acc_ts, kappa_ts, mf_ts = calculate_metrics(
        y_test,
        test_prediction
    )

    # --------------------------------------------------------
    # Store Results
    # --------------------------------------------------------

    dedicated_results.append({

        "ACC_TR": acc_tr,

        "K_TR": kappa_tr,

        "MF_TR": mf_tr,

        "ACC_TS": acc_ts,

        "K_TS": kappa_ts,

        "MF_TS": mf_ts

    })

    dedicated_parameters.append({

        "Repetition": rep + 1,

        "Lambda": BEST_LAMBDA,

        "C": BEST_C,

        "Loss": optimal_loss,

        "Theta_0": optimal_theta[0],
        "Theta_1": optimal_theta[1],
        "Theta_2": optimal_theta[2],
        "Theta_3": optimal_theta[3],
        "Theta_4": optimal_theta[4],
        "Theta_5": optimal_theta[5],
        "Theta_6": optimal_theta[6],
        "Theta_7": optimal_theta[7],
        "Theta_8": optimal_theta[8],
        "Theta_9": optimal_theta[9],
        "Theta_10": optimal_theta[10],
        "Theta_11": optimal_theta[11],
        "Theta_12": optimal_theta[12],
        "Theta_13": optimal_theta[13],
        "Theta_14": optimal_theta[14],
        "Theta_15": optimal_theta[15],
        "Theta_16": optimal_theta[16],
        "Theta_17": optimal_theta[17],
        "Theta_18": optimal_theta[18],
        "Theta_19": optimal_theta[19],
        "Theta_20": optimal_theta[20],
        "Theta_21": optimal_theta[21],
        "Theta_22": optimal_theta[22],
        "Theta_23": optimal_theta[23]

    })

    # --------------------------------------------------------
    # Runtime
    # --------------------------------------------------------

    elapsed = time.time() - start_time

    # --------------------------------------------------------
    # Print Results
    # --------------------------------------------------------

    print("\n" + "-" * 50)

    print("Training Metrics")

    print(f"ACC_TR : {acc_tr:.4f}")
    print(f"K_TR   : {kappa_tr:.4f}")
    print(f"MF_TR  : {mf_tr:.4f}")

    print()

    print("Testing Metrics")

    print(f"ACC_TS : {acc_ts:.4f}")
    print(f"K_TS   : {kappa_ts:.4f}")
    print(f"MF_TS  : {mf_ts:.4f}")

    print()

    print(f"Runtime : {elapsed/60:.2f} minutes")

    print("-" * 50)




DEDICATED (24) : REPETITION 1/5
Lambda : 0.01
C      : 100

Training Dedicated (24)...

Optimization Complete
Loss : 928.1047299665228
Theta Shape : 24

Computing Training Kernel...
Computing Test Kernel...

--------------------------------------------------
Training Metrics
ACC_TR : 0.9960
K_TR   : 0.9920
MF_TR  : 0.9960

Testing Metrics
ACC_TS : 0.9280
K_TS   : 0.8560
MF_TS  : 0.9280

Runtime : 13.09 minutes
--------------------------------------------------

DEDICATED (24) : REPETITION 2/5
Lambda : 0.01
C      : 100

Training Dedicated (24)...

Optimization Complete
Loss : 590.8386481115501
Theta Shape : 24

Computing Training Kernel...
Computing Test Kernel...

--------------------------------------------------
Training Metrics
ACC_TR : 1.0000
K_TR   : 1.0000
MF_TR  : 1.0000

Testing Metrics
ACC_TS : 0.9080
K_TS   : 0.8160
MF_TS  : 0.9080

Runtime : 13.03 minutes
--------------------------------------------------

DEDICATED (24) : REPETITION 3/5
Lambda : 0.01
C      : 100

Trainin

In [ ]:
# ============================================================
# BLOCK 6 : FINAL RESULTS
# Paper Format
# ============================================================

results_df = pd.DataFrame(dedicated_results)

parameters_df = pd.DataFrame(dedicated_parameters)

print("=" * 70)
print("Individual Results")
print("=" * 70)

display(results_df)

print("\n")

print("=" * 70)
print("Optimized Parameters")
print("=" * 70)

display(parameters_df)


# ============================================================
# Mean ± Standard Deviation
# ============================================================

def paper_format(column):

    mean = results_df[column].mean()

    std = results_df[column].std(ddof=1)

    return f"{mean:.2f} ({std:.2f})"


summary = pd.DataFrame({

    "Metric":[
        "ACC_TR",
        "κ_TR",
        "MF_TR",
        "ACC_TS",
        "κ_TS",
        "MF_TS"
    ],

    "Mean":[

        results_df["ACC_TR"].mean(),

        results_df["K_TR"].mean(),

        results_df["MF_TR"].mean(),

        results_df["ACC_TS"].mean(),

        results_df["K_TS"].mean(),

        results_df["MF_TS"].mean()

    ],

    "Std":[

        results_df["ACC_TR"].std(ddof=1),

        results_df["K_TR"].std(ddof=1),

        results_df["MF_TR"].std(ddof=1),

        results_df["ACC_TS"].std(ddof=1),

        results_df["K_TS"].std(ddof=1),

        results_df["MF_TS"].std(ddof=1)

    ]

})

print("\n")
print("=" * 70)
print("Summary Statistics")
print("=" * 70)

display(summary.round(4))


# ============================================================
# TABLE 2 FORMAT
# ============================================================

paper_table = pd.DataFrame({

    "Dataset":[
        "MNIST-PCA-8"
    ],

    "Mapping":[
        "ZZFeatureMap"
    ],

    "QKT approach":[
        "Dedicated (24)"
    ],

    "ACC_TR":[
        paper_format("ACC_TR")
    ],

    "κ_TR":[
        paper_format("K_TR")
    ],

    "MF_TR":[
        paper_format("MF_TR")
    ],

    "ACC_TS":[
        paper_format("ACC_TS")
    ],

    "κ_TS":[
        paper_format("K_TS")
    ],

    "MF_TS":[
        paper_format("MF_TS")
    ]

})

print("\n")
print("=" * 70)
print("TABLE 2 FORMAT")
print("=" * 70)

display(paper_table)

Individual Results


,ACC_TR,K_TR,MF_TR,ACC_TS,K_TS,MF_TS
0,0.996,0.992,0.996,0.928,0.856,0.927995
1,1.000,1.000,1.000,0.908,0.816,0.907963
2,0.996,0.992,0.996,0.924,0.848,0.923970
3,1.000,1.000,1.000,0.944,0.888,0.943943
4,1.000,1.000,1.000,0.952,0.904,0.951997




Optimized Parameters


,Repetition,Lambda,C,Loss,Theta_0,Theta_1,Theta_2,Theta_3,Theta_4,Theta_5,...,Theta_14,Theta_15,Theta_16,Theta_17,Theta_18,Theta_19,Theta_20,Theta_21,Theta_22,Theta_23
0,1,0.01,100,928.104730,3.010293,1.524347,-0.211291,2.565958,-2.015315,0.849941,...,0.473622,-1.424288,0.561146,-2.247292,0.620920,2.028348,0.434692,-1.156194,-0.444619,0.506067
1,2,0.01,100,590.838648,-2.385889,-0.203547,3.468223,-1.923029,-0.441393,1.568527,...,0.363765,0.115515,-2.318398,1.971723,-2.076590,-0.431931,-2.768404,-1.836175,-0.186322,1.812354
2,3,0.01,100,1081.110715,-0.566757,-0.573252,-0.693482,-1.985854,0.538539,0.260409,...,0.135032,-0.345920,-0.814476,-0.568716,-0.880964,0.901903,0.259286,-0.659362,0.533445,-1.584018
3,4,0.01,100,396.174080,-2.450747,2.464400,2.287229,-1.946118,-1.513447,-1.093884,...,-3.056742,-1.520397,-3.570535,-3.541397,-2.281602,-2.237961,-0.802340,-1.514329,-2.048698,3.365818
4,5,0.01,100,520.539815,-2.165862,0.098034,2.793054,1.677602,2.208572,-0.097480,...,-1.237662,2.050129,3.922440,0.244203,-1.068890,-0.622094,-1.227463,0.792956,2.257989,-2.201645




Summary Statistics


,Metric,Mean,Std
0,ACC_TR,0.9984,0.0022
1,κ_TR,0.9968,0.0044
2,MF_TR,0.9984,0.0022
3,ACC_TS,0.9312,0.0173
4,κ_TS,0.8624,0.0346
5,MF_TS,0.9312,0.0173




TABLE 2 FORMAT


,Dataset,Mapping,QKT approach,ACC_TR,κ_TR,MF_TR,ACC_TS,κ_TS,MF_TS
0,MNIST-PCA-8,ZZFeatureMap,Dedicated (24),1.00 (0.00),1.00 (0.00),1.00 (0.00),0.93 (0.02),0.86 (0.03),0.93 (0.02)
